# Delineate shapfile using web-based API and save as GeoJSON files.

In [17]:
import os
import time
import requests
import pandas as pd
import json

# 1. Load CSV
csv_file = "rawdata/selected_stations.csv"
df = pd.read_csv(csv_file)

# 2. Output folder
output_folder = "rawdata/web_based"
os.makedirs(output_folder, exist_ok=True)

# 3. ✅  API endpoint
API_URL = "https://mghydro.com/app/watershed_api"

print(f"Loaded {len(df)} points. Starting delineation...\n")

for i, row in df.iterrows():

    lat = float(row["Lat"])
    lng = float(row["Lon"])   # IMPORTANT: API uses 'lng', not 'lon'

    name = row.get("Station_ID", f"station_{i}")
    point_id = row.get("Name", f"basin_{name}")

    output_file = os.path.join(output_folder, f"{point_id}.geojson")

    print(f"[{i+1}/{len(df)}] Processing {point_id} ({lat}, {lng})")

    params = {
        "lat": lat,
        "lng": lng,
        "precision": "high",   # optional but better quality
        "simplify": "true"
    }

    try:
        r = requests.get(API_URL, params=params, timeout=120)

        print("   Status:", r.status_code)

        if r.status_code != 200:
            print("   ❌ Error:", r.text[:300])
            continue

        # Must be GeoJSON
        data = r.json()

        with open(output_file, "w") as f:
            json.dump(data, f)

        print(f"   ✅ Saved -> {output_file}")

    except Exception as e:
        print("   ❌ Failed:", e)

    time.sleep(3)  # important (API rate limit warning)

Loaded 24 points. Starting delineation...

[1/24] Processing basin_406.5 (28.22, 83.64)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_406.5.geojson
[2/24] Processing basin_438.0 (28.1, 84.23)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_438.0.geojson
[3/24] Processing basin_120.0 (29.67, 80.56)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_120.0.geojson
[4/24] Processing basin_610.0 (27.75, 85.85)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_610.0.geojson
[5/24] Processing basin_289.95 (28.34, 81.74)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_289.95.geojson
[6/24] Processing basin_589.0 (27.11, 85.46)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_589.0.geojson
[7/24] Processing basin_439.7 (27.95, 84.43)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_439.7.geojson
[8/24] Processing basin_670.0 (27.27, 86.66)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_670.0.geojson
[9/24] Processing basin_447.0 (27.97, 85.18)
   Status: 200


# For all delineated catchment, calculate the drainage area in km2 and save to a csv file.

In [18]:
import os
import geopandas as gpd
import pandas as pd

geojson_folder = "rawdata/web_based"
output_folder = 'rawdata/web_based'

results = []

print("Computing catchment areas...\n")

for file in os.listdir(geojson_folder):
    if not file.endswith(".geojson"):
        continue

    path = os.path.join(geojson_folder, file)

    try:
        gdf = gpd.read_file(path)

        # merge geometry in case of multiple features
        geom = gdf.geometry.unary_union

        basin_id = file.replace(".geojson", "")

        # convert to GeoDataFrame
        gdf_single = gpd.GeoDataFrame([1], geometry=[geom], crs="EPSG:4326")

        # project to equal-area CRS
        gdf_proj = gdf_single.to_crs(epsg=6933)

        area_m2 = gdf_proj.geometry.area.values[0]
        area_km2 = area_m2 / 1e6

        results.append({
            "basin_id": basin_id,
            "area_km2": area_km2
        })

        print(f"{basin_id}: {area_km2:.2f} km²")

    except Exception as e:
        print(f"Failed {file}: {e}")

# final dataframe
df = pd.DataFrame(results)
# round to 0 decimal
df["area_km2"] = df["area_km2"].round(0)

df.to_csv(f"{output_folder}/catchment_areas.csv", index=False)

print("\nSaved: catchment_areas.csv")

Computing catchment areas...

basin_120.0: 1185.63 km²
basin_215.0: 16970.91 km²
basin_259.2: 4040.05 km²
basin_260.0: 7075.96 km²
basin_270.0: 13455.98 km²
basin_280.0: 45458.12 km²
basin_289.95: 2582.61 km²
basin_375.0: 8.43 km²
basin_406.5: 6586.75 km²
basin_419.1: 10622.94 km²
basin_420.0: 11815.28 km²
basin_438.0: 852.45 km²
basin_439.7: 4046.90 km²
basin_445.0: 3943.06 km²
basin_447.0: 4625.98 km²
basin_450.0: 31683.37 km²
basin_589.0: 51.55 km²
basin_604.5: 27486.45 km²
basin_610.0: 2534.43 km²
basin_630.0: 4841.81 km²
basin_652.0: 10161.42 km²
basin_670.0: 3716.80 km²
basin_690.0: 5889.96 km²
basin_695.0: 54044.55 km²

Saved: catchment_areas.csv


C:\Users\sp2596\AppData\Local\Temp\ipykernel_26628\2790172253.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_26628\2790172253.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_26628\2790172253.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_26628\2790172253.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_26628\2790172253.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.g